# Calibrate AnomalyCLIP decision thresholds

The official AnomalyCLIP paper and repository do not publish a binary decision threshold; official clean results use threshold-independent AUROC/AP/AUPRO. This notebook creates a **custom, explicitly labeled fallback** for secondary metrics such as flip rate and targeted success. For each dataset and category, it sets the threshold to the 95th percentile of AnomalyCLIP scores on clean normal training images only. Test images, test labels, anomalies, and adversarial images are never used for calibration.

In [ ]:
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE REPOSITORIES AND INSTALL DEPENDENCIES =====')
WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
ANOMALYCLIP_ROOT = WORKING / 'AnomalyCLIP'
EXPERIMENT_REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
ANOMALYCLIP_REPO_URL = 'https://github.com/zqhang/AnomalyCLIP.git'
ANOMALYCLIP_COMMIT = '3911738c0867544f545a076ad78f3f11d9ecbfdf'

def clone_or_update(url, destination, commit=None):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    else:
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    if commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)

clone_or_update(EXPERIMENT_REPO_URL, EXPERIMENT_ROOT)
clone_or_update(ANOMALYCLIP_REPO_URL, ANOMALYCLIP_ROOT, ANOMALYCLIP_COMMIT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'requirements.txt')
], check=True)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))
print('Experiment code:', EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline')
print('Official AnomalyCLIP:', ANOMALYCLIP_ROOT)

In [ ]:
import torch

print('===== STEP 2: RESOLVE DATASETS AND CHECKPOINTS =====')

def first_existing_directory(paths, label):
    for path in paths:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'{label} was not found. Checked: {paths}')

MVTEC_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
], 'MVTec AD')
VISA_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'),
    Path('/kaggle/input/visa-ad/VisA_20220922'),
], 'VisA')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')

# These are the same opposite-dataset checkpoints used by the evaluator.
MVTEC_TARGET_CHECKPOINT = ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale' / 'epoch_15.pth'
VISA_TARGET_CHECKPOINT = ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale_visa' / 'epoch_15.pth'
for checkpoint in (MVTEC_TARGET_CHECKPOINT, VISA_TARGET_CHECKPOINT):
    if not checkpoint.is_file():
        available = sorted((ANOMALYCLIP_ROOT / 'checkpoints').rglob('*.pth'))
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint}. Available: {available}')

MODEL_KWARGS_BY_TARGET = {
    'mvtec': {
        'repository_root': str(ANOMALYCLIP_ROOT),
        'checkpoint_path': str(MVTEC_TARGET_CHECKPOINT),
        'clip_download_root': str(WORKING / 'clip_cache'),
    },
    'visa': {
        'repository_root': str(ANOMALYCLIP_ROOT),
        'checkpoint_path': str(VISA_TARGET_CHECKPOINT),
        'clip_download_root': str(WORKING / 'clip_cache'),
    },
}
print('MVTec:', MVTEC_ROOT)
print('VisA:', VISA_ROOT)

In [ ]:
from blackbox_evaluation_pipeline import ThresholdCalibrationConfig, calibrate_thresholds

print('===== STEP 3: CALIBRATE ON CLEAN NORMAL TRAINING IMAGES =====')
DATASETS = ('mvtec', 'visa')
THRESHOLD_QUANTILE = 0.95
OUTPUT_ROOT = WORKING / 'anomalyclip_thresholds_q95'

config = ThresholdCalibrationConfig(
    output_root=str(OUTPUT_ROOT),
    model_name='anomalyclip',
    model_kwargs_by_target=MODEL_KWARGS_BY_TARGET,
    datasets=DATASETS,
    mvtec_root=str(MVTEC_ROOT),
    visa_root=str(VISA_ROOT),
    device='cuda',
    batch_size=2,
    image_size=518,
    quantile=THRESHOLD_QUANTILE,
    provenance='custom_q95_fallback_official_anomalyclip_has_no_decision_threshold',
    official_model_threshold=False,
    run_metadata={
        'anomalyclip_commit': ANOMALYCLIP_COMMIT,
        'official_repository': ANOMALYCLIP_REPO_URL,
        'calibration_uses_test_data': False,
        'calibration_uses_anomaly_data': False,
    },
)
GENERATED_THRESHOLDS = calibrate_thresholds(config)

In [ ]:
import json

print('===== STEP 4: PRINT GENERATED THRESHOLDS =====')
for dataset, threshold_path in GENERATED_THRESHOLDS.items():
    payload = json.loads(threshold_path.read_text(encoding='utf-8'))
    print(f'\n[{dataset}] {threshold_path}')
    print('category | threshold | count | min | mean | max | std')
    for category, record in payload['categories'].items():
        print(
            f"{category:12s} | {record['threshold']:.6f} | "
            f"{record['sample_count']:5d} | {record['score_min']:.6f} | "
            f"{record['score_mean']:.6f} | {record['score_max']:.6f} | "
            f"{record['score_std']:.6f}"
        )

In [ ]:
import shutil

print('===== STEP 5: PACKAGE THRESHOLD ARTIFACTS =====')
archive = shutil.make_archive(
    str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
print('Packaged thresholds:', archive)